# 13 — Validate phenotype-stratified antibiogram reporting

**Project:** Clinical Microbiology Analytics Lakehouse  
**Source:** ARMD release `2025-04-11`  
**Warehouse:** `workspace.micro_dev`

This notebook records the independently selected reporting populations and the effect of a 12-month shifted-date window. It preserves eight SQL cells and their saved outputs.

**The saved results span two runs.** Sections 1–6 contain the full-release phenotype-stratified baseline (455 mart rows, 381 displayable). Section 7 contains the later 2023 coverage outputs (424 rows, 194 displayable), including explicit window columns. Read each output with its scope label; executing every cell now would query the currently built 2023 tables, not recreate the stored full-release baseline.

## Reporting rules and execution context

- Ordinary (`base`), ESCE-R, VRE, MRSA, and MSSA categories independently select the first qualifying culture per patient/cohort/organism/category. The same patient can appear in several categories; their denominators are not additive.
- Overall *S. aureus* selects the first culture irrespective of phenotype. MRSA/MSSA do not need to be mutually exclusive patient populations.
- CRE/MDR reporting records are excluded from the main susceptibility categories and summarized in the separate MDRO mart. `base` does not establish wild-type susceptibility.
- The reporting window is applied before culture ranking and MDRO aggregation. Both marts publish the inclusive start and exclusive end dates.
- Display requires at least **30 categorized S/I/R endpoints** for that organism/category/drug/cohort. This is a project reporting rule, not a claim of guideline certification.

The current configuration is `analysis_window_start: '2023-01-01'`, with end `2024-01-01` exclusive. Use the Databricks-connected notebook kernel; each code cell starts with `%sql` and contains one SQL statement. Full-release comparisons require a separately built and preserved full-release scope.

The saved full `dbt build` completed on **October 12, 2026, at 10:48 p.m. Eastern** (`2026-10-13T02:48:09Z`): **22 models, 4 reference seeds, and 224 passing data tests** — **250 successful operations**, with no error, warning, or skipped result statuses. The saved compiled manifest uses the 2023 window. These are recorded build results; the documentation revision did not execute a new warehouse build.

### 1. Inspect reporting categories

**Purpose:** List organism/reporting-phenotype categories actually appearing in `mart_susceptibility`, including their row counts and largest eligible first-culture population.

**Interpretation:** The main susceptibility mart should not contain the CRE or MDR *P. aeruginosa* categories. The `overall` category should be reserved for *Staphylococcus aureus*; other organism populations are stratified into `base` and qualifying phenotype categories. This is an **inventory query**, not a standalone assertion; it only displays categories named in its SQL filter.

**Saved output scope:** full-release phenotype-stratified baseline, before activating the 2023 window. This is historical comparison evidence, not the 2023 population.

The stored inventory displays the first 20 rows. Its phenotype filter and truncated output cannot prove that every unexpected category is absent. Use the compact complete summary below and the dbt category/completeness assertions.

In [4]:
%sql
select reporting_phenotype, organism_reporting_group,
       count(*) as mart_rows,
       max(eligible_count) as maximum_eligible_count
from workspace.micro_dev.mart_susceptibility
where reporting_phenotype in ('overall','esce_r','base',
  'oxacillin_resistant','oxacillin_susceptible','vancomycin_resistant',
  'cre','mdr_p_aeruginosa')
group by reporting_phenotype, organism_reporting_group
order by reporting_phenotype, organism_reporting_group;

,reporting_phenotype,organism_reporting_group,mart_rows,maximum_eligible_count
0,base,Citrobacter freundii complex,18,860
1,base,Citrobacter koseri,23,1004
2,base,Coagulase-negative staphylococci (unspeciated),3,1247
3,base,Enterobacter cloacae complex,12,927
4,base,Enterococcus faecalis,15,867
5,base,Enterococcus faecium,15,135
6,base,Enterococcus species (unspeciated),2,7354
7,base,Escherichia coli,21,34714
8,base,Klebsiella aerogenes,20,826
9,base,Klebsiella oxytoca,18,962


### 2. Quantify overlap across separate phenotype reporting populations

**Purpose:** Count distinct patients represented in the `base`, ESCE-R, and VRE reporting categories for each cohort and organism.

**Interpretation:** A positive `patients_in_both_categories` is **expected and allowed**. A patient can contribute one first culture to `base` and another first qualifying culture to ESCE-R or VRE. Do not sum these populations to estimate unique patients. This query does not measure MRSA/MSSA overlap.

**Saved output scope:** full-release phenotype-stratified baseline, before activating the 2023 window. This is historical comparison evidence, not the 2023 population.

Example: the stored urine E. coli baseline has 34,714 base patients, 3,500 ESCE-R patients, and 1,119 in both. The displayed result is limited to 20 rows; it is not a complete list of organism/cohort combinations.

In [5]:
%sql
with labels as (
    select source_version, cohort_id, organism_reporting_group,
           patient_key,
           max(case when reporting_phenotype='base' then 1 else 0 end) as base_patient,
           max(case when reporting_phenotype='esce_r' then 1 else 0 end) as esce_patient,
           max(case when reporting_phenotype='vancomycin_resistant' then 1 else 0 end) as vre_patient
    from workspace.micro_dev.int_first_stratified_reporting_culture
    group by source_version, cohort_id, organism_reporting_group, patient_key
)
select cohort_id, organism_reporting_group,
       sum(base_patient) as base_patients,
       sum(esce_patient) as esce_r_patients,
       sum(vre_patient) as vre_patients,
       sum(case when base_patient=1 and (esce_patient=1 or vre_patient=1)
           then 1 else 0 end) as patients_in_both_categories
from labels
group by cohort_id, organism_reporting_group
having sum(esce_patient)+sum(vre_patient)>0
order by patients_in_both_categories desc, cohort_id, organism_reporting_group;

,cohort_id,organism_reporting_group,base_patients,esce_r_patients,vre_patients,patients_in_both_categories
0,urine,Escherichia coli,34714,3500,0,1119
1,urine,Klebsiella pneumoniae,6114,393,0,96
2,blood,Escherichia coli,2565,495,0,41
3,urine,Enterobacter cloacae complex,927,184,0,28
4,urine,Klebsiella aerogenes,826,103,0,11
5,urine,Klebsiella oxytoca,962,66,0,11
6,respiratory,Escherichia coli,320,67,0,7
7,blood,Enterococcus faecium,135,0,139,5
8,urine,Enterococcus faecalis,867,0,8,4
9,urine,Enterococcus faecium,125,0,77,4


### 3. Check *Staphylococcus aureus* populations

**Purpose:** Report the eligible first-culture denominator separately for `overall`, `oxacillin_resistant` (MRSA), and `oxacillin_susceptible` (MSSA) within each specimen cohort.

**Interpretation:** `overall` is the first *S. aureus* culture irrespective of phenotype; MRSA and MSSA are selected independently as first qualifying cultures. The query uses `max(eligible_count)` because the population count is repeated across the approved antibiotics in each category.

**Saved output scope:** full-release phenotype-stratified baseline, before activating the 2023 window. This is historical comparison evidence, not the 2023 population.

For blood, 546 MRSA + 1,206 MSSA exceeds 1,737 overall patients. This is allowed across independently selected categories; it does not relax within-category uniqueness.

In [6]:
%sql
select cohort_id, reporting_phenotype, max(eligible_count) as n_first_cultures
from workspace.micro_dev.mart_susceptibility
where organism_reporting_group='Staphylococcus aureus'
group by cohort_id, reporting_phenotype
order by cohort_id, reporting_phenotype;

,cohort_id,reporting_phenotype,n_first_cultures
0,blood,overall,1737
1,blood,oxacillin_resistant,546
2,blood,oxacillin_susceptible,1206
3,respiratory,overall,3430
4,respiratory,oxacillin_resistant,618
5,respiratory,oxacillin_susceptible,2931
6,urine,overall,517
7,urine,oxacillin_resistant,174
8,urine,oxacillin_susceptible,331


### 4. Compare MDRO surveillance totals

**Purpose:** Summarize `mart_mdro` row counts, eligible patient counts, and phenotype-positive patient counts by cohort and MDRO definition.

**Interpretation:** In **full-release mode**, compare these numbers with a saved, same-release baseline. They should remain unchanged by susceptibility-category restructuring. If a restricted reporting window is active, differences are expected. The MDRO mart has a separate any-positive-across-observations methodology.

**Saved output scope:** full-release phenotype-stratified baseline, before activating the 2023 window. This is historical comparison evidence, not the 2023 population.

The six saved summary rows contain 31 CRE mart rows and three MDR P. aeruginosa rows, **34 MDRO rows in total**. A prior 31-row combined summary omitted the MDR rows. Summing `patients` across base organisms counts patient–organism groups, not necessarily distinct patients across the whole cohort. Current-window MDRO totals require a same-window comparison.

In [7]:
%sql
-- relative to the last known validated baseline.
select mdro_phenotype, cohort_id, count(*) as rows,
       sum(denominator_count) as patients,
       sum(positive_count) as positive_patients
from workspace.micro_dev.mart_mdro
group by mdro_phenotype, cohort_id
order by mdro_phenotype, cohort_id;

,mdro_phenotype,cohort_id,rows,patients,positive_patients
0,CRE,blood,10,5203,27
1,CRE,respiratory,10,1911,17
2,CRE,urine,11,52732,130
3,MDR Pseudomonas aeruginosa,blood,1,384,24
4,MDR Pseudomonas aeruginosa,respiratory,1,1889,254
5,MDR Pseudomonas aeruginosa,urine,1,1956,72


### 5. Inspect shared reporting AST reconciliation

**Purpose:** Inspect the `ast_scope` categories, endpoint volumes, reporting source groups, and conflict counts in `int_culture_reporting_susceptibility`.

**Interpretation:** This descriptive check makes the provenance of selected AST observations visible. Review whether the scopes reflect the new phenotype-stratified design and whether conflicts are retained rather than silently assigned a result.

**Saved output scope:** full-release phenotype-stratified baseline, before activating the 2023 window. This is historical comparison evidence, not the 2023 population.

The saved 98,929 overall and 1,523,124 stratified endpoints describe different AST source scopes before final first-culture selection. They are not comparable patient denominators and should not be added as unique tested observations.

In [8]:
%sql
select ast_scope, count(*) as endpoint_rows,
       count(distinct reporting_source_key) as culture_groups,
       sum(case when resolution_status='conflict' then 1 else 0 end) as conflicting_endpoints
from workspace.micro_dev.int_culture_reporting_susceptibility
group by ast_scope;

,ast_scope,endpoint_rows,culture_groups,conflicting_endpoints
0,overall,98929,9906,771
1,stratified,1523124,123828,9557


## 6. Complete reporting-category summary — full-release baseline

This compact query has no phenotype filter and its saved output contains all six categories. The totals are **455 mart rows and 381 displayable rows**. No CRE or MDR category appears. The query does not identify which organism occupies each category; use the detailed inventory and dbt assertions to establish that `overall` is limited to *S. aureus*.

In [9]:
%sql
select
    reporting_phenotype,
    count(*) as mart_rows,
    sum(case
        when display_status = 'display' then 1
        else 0
    end) as displayable_rows
from workspace.micro_dev.mart_susceptibility
group by reporting_phenotype
order by reporting_phenotype;

,reporting_phenotype,mart_rows,displayable_rows
0,base,287,264
1,esce_r,89,53
2,overall,18,18
3,oxacillin_resistant,18,18
4,oxacillin_susceptible,18,18
5,vancomycin_resistant,25,10


## 7. Twelve-month reporting coverage

**Saved output scope: 2023 shifted-date window.** The following query counts reporting rows by cohort and phenotype, split into displayable and insufficient-data pairs. The second query repeats the calculation with the mart's window columns so the scope is explicit.

`pct_displayable` is the percentage of reporting rows meeting the count rule. It is not the percentage of patients tested or the percentage susceptible. A retained insufficient-data row still has an eligible population; its categorized AST count is below 30. A category with no eligible selected population may have no mart rows at all.

In [10]:
%sql
select
    cohort_id,
    reporting_phenotype,
    count(*) as total_pairs,
    sum(
        case
            when display_status = 'display' then 1
            else 0
        end
    ) as displayable_pairs,
    sum(
        case
            when display_status = 'insufficient_data' then 1
            else 0
        end
    ) as insufficient_data_pairs,
    round(
        100.0 * sum(
            case
                when display_status = 'display' then 1
                else 0
            end
        ) / nullif(count(*), 0),
        1
    ) as pct_displayable
from workspace.micro_dev.mart_susceptibility
group by
    cohort_id,
    reporting_phenotype
order by
    cohort_id,
    reporting_phenotype;

,cohort_id,reporting_phenotype,total_pairs,displayable_pairs,insufficient_data_pairs,pct_displayable
0,blood,base,100,35,65,35.0
1,blood,esce_r,30,6,24,20.0
2,blood,overall,6,6,0,100.0
3,blood,oxacillin_resistant,6,6,0,100.0
4,blood,oxacillin_susceptible,6,6,0,100.0
5,blood,vancomycin_resistant,10,0,10,0.0
6,respiratory,base,75,14,61,18.7
7,respiratory,esce_r,30,0,30,0.0
8,respiratory,overall,6,6,0,100.0
9,respiratory,oxacillin_resistant,6,6,0,100.0


### Confirm the window carried by the mart

The following saved result includes the start and exclusive end dates. It confirms that the coverage counts above belong to the 2023 build. Grouping by both dates would expose multiple recorded windows if they were present; it does not replace row-level date-selection tests.

In [11]:
%sql
select
    analysis_window_start,
    analysis_window_end_exclusive,
    cohort_id,
    reporting_phenotype,
    count(*) as total_pairs,
    sum(case
        when display_status = 'display' then 1
        else 0
    end) as displayable_pairs,
    sum(case
        when display_status = 'insufficient_data' then 1
        else 0
    end) as insufficient_data_pairs,
    round(
        100.0 * sum(case
            when display_status = 'display' then 1
            else 0
        end) / nullif(count(*), 0),
        1
    ) as pct_displayable
from workspace.micro_dev.mart_susceptibility
group by
    analysis_window_start,
    analysis_window_end_exclusive,
    cohort_id,
    reporting_phenotype
order by
    cohort_id,
    reporting_phenotype;

,analysis_window_start,analysis_window_end_exclusive,cohort_id,reporting_phenotype,total_pairs,displayable_pairs,insufficient_data_pairs,pct_displayable
0,2023-01-01,2024-01-01,blood,base,100,35,65,35.0
1,2023-01-01,2024-01-01,blood,esce_r,30,6,24,20.0
2,2023-01-01,2024-01-01,blood,overall,6,6,0,100.0
3,2023-01-01,2024-01-01,blood,oxacillin_resistant,6,6,0,100.0
4,2023-01-01,2024-01-01,blood,oxacillin_susceptible,6,6,0,100.0
5,2023-01-01,2024-01-01,blood,vancomycin_resistant,10,0,10,0.0
6,2023-01-01,2024-01-01,respiratory,base,75,14,61,18.7
7,2023-01-01,2024-01-01,respiratory,esce_r,30,0,30,0.0
8,2023-01-01,2024-01-01,respiratory,overall,6,6,0,100.0
9,2023-01-01,2024-01-01,respiratory,oxacillin_resistant,6,6,0,100.0


## 8. Recorded 2023 result and interpretation

The dated coverage output confirms **2023-01-01 inclusive through 2024-01-01 exclusive** for all 17 returned cohort/category groups.

| Cohort | Mart rows | Displayable | Insufficient data | Displayable (%) |
|---|---:|---:|---:|---:|
| Blood | 158 | 59 | 99 | 37.3% |
| Respiratory | 123 | 32 | 91 | 26.0% |
| Urine | 143 | 103 | 40 | 72.0% |
| **Total** | **424** | **194** | **230** | **45.8%** |

The 424 total rows reconcile to 194 displayable + 230 insufficient-data rows. Compared with the saved full-release baseline's 381 displayable rows, the annual window retains 194: 187 fewer, a 49.1% reduction.

| Reporting population | Full-release rows | Full-release displayable | 2023 rows | 2023 displayable |
|---|---:|---:|---:|---:|
| Ordinary (`base`) | 287 | 264 | 266 | 129 |
| ESCE-R | 89 | 53 | 89 | 19 |
| Overall *S. aureus* | 18 | 18 | 18 | 18 |
| MRSA | 18 | 18 | 18 | 12 |
| MSSA | 18 | 18 | 18 | 16 |
| VRE | 25 | 10 | 15 | 0 |
| **Total** | **455** | **381** | **424** | **194** |

Urine has the broadest coverage, with 80/91 ordinary/base pairs displayable. Overall *S. aureus* retains all six approved antibiotics in every cohort. Blood and respiratory MRSA/MSSA each retain six; urine MRSA retains none and urine MSSA retains four of six. ESCE-R retains six blood, zero respiratory, and 13 urine pairs. VRE retains no displayable percentages; this does not establish absence of VRE.

Keep the minimum-30 rule and the insufficient-data records for audit. The 2023 demonstration should clearly label low-count percentages as **Insufficient data**, while full-release results remain a separately labeled comparison. These inventories complement the dbt assertions; they do not independently prove every selected culture or validate a 2023 MDRO reconstruction.

### Publication checkpoint

Use matching source release, model version, window, and reference seeds for each reporting export. Confirm the window fields and display status before refreshing Tableau. Notebook 12 now reconstructs MDRO evidence within the date window recorded in the built mart, before patient aggregation. Rerun its revised section 2 and save the dated result to complete the independent MDRO validation. The notebook cleanup did not execute a new warehouse build or change the saved outputs in this notebook.


### Scoped exclusion update — October 13, 2026

The saved susceptibility results above precede 28 newly approved cohort/category exclusions. Applying those rules to the saved 2023 export projects **396 reporting rows: 187 displayable and 209 insufficient-data**. Keep the saved 424-row output as the pre-exclusion baseline until a fresh dbt build and notebook execution record the revised counts. The MDRO mart does not use this exclusion mapping.
